# Advanced IR Project Part 1

Version 1.1

In the Project Part 1 assessment item of Advanced IR you will build a cascade ranking model for a new test collection.

#### Grading

This project is assessed using the following criteria (20pts in total):

**Implementation (10pts)**
 - Code quality and comments (2.5pts)
 - Ranking cascade pipeline of at least two ranking models (2.5pts)
 - Comparison to a BM25 baseline (5pts)

**Discussion (10pts)**
 - Presentation, spelling, and grammar (5pts)
 - Appropriate and thoughtful responses to prompts (5pts)

#### Submission

Export the notebook as a PDF document with all of the code cells executed to show the outputs and the discussion filled out. Submit the PDF of this notebook to ILIAS ([project link](https://ovidius.uni-tuebingen.de/ilias3/ilias.php?baseClass=ilexercisehandlergui&cmdNode=cp:o6&cmdClass=ilObjExerciseGUI&cmd=showOverview&ref_id=5379322)) by the 18.11.2025 23:55.

In [1]:
!pip install lsr-benchmark ir-datasets lightning_ir
# Use this cell to install your other dependencies.

In [2]:
import ir_datasets
from lsr_benchmark import register_to_ir_datasets

from pyterrier.datasets import get_dataset
register_to_ir_datasets("msmarco-passage/trec-dl-2020/judged")
dataset = get_dataset(f"irds:lsr-benchmark/msmarco-passage/trec-dl-2020/judged")

## Implementation

Use as many cells as you like below to implement your cascade ranking model on the dataset provided above. Your cascade should compose **at least two** ranking models. Once you've implemented your model compare it to BM25 by choosing appropriate evaluation measures.

In [3]:
for doc in dataset.get_corpus_iter():
    print(doc["text"])
    break

lsr-benchmark/msmarco-passage/trec-dl-2020/judged documents:   0%|          | 0/49614 [00:00<?, ?it/s]

Definition of Medical Products. Medical products, for the purposes of this questionnaire, are defined as selected products used to diagnose, treat or care for patients. This excludes electronic medical products (anything that plugs in or has a battery). Greening Supply Chain Series: Bisphenol A (BPA) in Medical...


In [4]:
# Colab: Add the current directory ('.') to the list of paths where Python looks for modules
import sys
import os
sys.path.append(os.path.abspath('.'))

In [5]:
import torch
import pyterrier as pt
import pandas as pd

from lightning_ir import TorchDenseIndexConfig, TorchDenseSearchConfig
from lightning_ir_pyterrier import BiEncoder, CrossEncoder
from pathlib import Path
from pyterrier.text import get_text

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
if torch.backends.mps.is_available():
    device = "mps"

/home/kpch/miniconda3/envs/air/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/kpch/miniconda3/envs/air/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [6]:
dataset_name = "trec-dl-2020"
index_dir = Path(f"./vector_index_{dataset_name.replace('/','-')}")

# Load the dataset that has the FULL CORPUS 
corpus_dataset = pt.get_dataset('irds:msmarco-passage')

print("Triggering full corpus download (if needed)...")
next(iter(corpus_dataset.get_corpus_iter()))
print("Full corpus is ready.")

Triggering full corpus download (if needed)...


msmarco-passage documents:   0%|          | 0/8841823 [00:00<?, ?it/s]

Full corpus is ready.


In [7]:
K_EVAL = 100                    # Cut-off 'k' we will evaluate at
K_STAGE1_CANDIDATES = 1000      # BM25 retrieves 1000
K_STAGE2_CANDIDATES = K_EVAL    # 100 for the final stage

In [8]:
# # Bi-Encoder
# bi_encoder = BiEncoder(
#     model_name_or_path="webis/bert-bi-encoder",
#     index_dir=index_dir,
#     index_config=TorchDenseIndexConfig(),
#     search_config=TorchDenseSearchConfig(k=100),
#     batch_size=32,
#     verbose=True,
#     use_gpu=True
# )
# bi_encoder._module.to(device)
# bi_encoder._module.device

# if not index_dir.exists():
#     print("Building dense (Bi-Encoder) index from full corpus...")
#     # Index the FULL corpus, not just the 'judged' subset
#     bi_encoder.index(corpus_dataset.get_corpus_iter())

In [9]:
# Cross-Encoder
cross_encoder = CrossEncoder(
    model_name_or_path="webis/monoelectra-base",
    batch_size=1,
    verbose=True,
    use_gpu=True
)
if torch.cuda.is_available():
    cross_encoder._module.to(device)
    cross_encoder._module.device

You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'MonoElectraTokenizerFast'.


In [ ]:
pt_index_path = f"./index_{dataset_name.replace('/','-')}"

if not os.path.exists(pt_index_path + "/data.properties"):
  # create the index, using the IterDictIndexer indexer
  indexer = pt.index.IterDictIndexer(
      pt_index_path, 
      meta_reverse=[],
      fields=["text"]  
  )
  index_ref = indexer.index(corpus_dataset.get_corpus_iter())
else:
  index_ref = pt_index_path

index = pt.IndexFactory.of(index_ref)
bm25 = pt.BatchRetrieve(index, wmodel="BM25")

Java started (triggered by IndexFactory.of) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_1466964/904546704.py:15: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25 = pt.BatchRetrieve(index, wmodel="BM25")


In [11]:
# Create a baseline that is truncated to the same cut-off of K_EVAL for better evaluation analysis
bm25_baseline_at_k = (pt.BatchRetrieve(index, wmodel="BM25") % K_EVAL)

/tmp/ipykernel_1466964/3323897820.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25_baseline_at_k = (pt.BatchRetrieve(index, wmodel="BM25") % K_EVAL)


In [12]:
# Cascade Pipeline (BM25 >> cross_encoder)
# Stage 1: BM25 retrieves 1000 candidates
bm25_stage1 = pt.BatchRetrieve(index, wmodel="BM25") % K_STAGE1_CANDIDATES

# Stage 2: Cross-Encoder re-ranks the 1000, and we take the final 100
# Note: K_STAGE2_CANDIDATES (100) is applied after the cross-encoder
cascade_model = (
    bm25_stage1
    >> get_text(corpus_dataset) 
    >> cross_encoder
) % K_STAGE2_CANDIDATES

/tmp/ipykernel_1466964/4091805109.py:3: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25_stage1 = pt.BatchRetrieve(index, wmodel="BM25") % K_STAGE1_CANDIDATES


In [13]:
from ir_measures import nDCG, RR, MAP

# nDCG@10: Quality of the all-important first page
# nDCG@100: Quality of the entire list we are comparing
# MAP@100: A cut-off version of MAP
# RR: Rank of the first relevant item

topics = dataset.get_topics(variant='text')

print(f"Running experiment on {len(topics)} topics, evaluating at k={K_EVAL}...")
experiment_results_at_k = pt.Experiment(
    [
        bm25_baseline_at_k,
        cascade_model
    ],
    dataset.get_topics(variant='text'),  
    dataset.get_qrels(),   
    eval_metrics=[nDCG@10, RR, nDCG@100, MAP@100],
    names=["BM25 Baseline at 100", "Cascade Model (BM25 >> Cross-Encoder)"],
    verbose=True,
)

Running experiment on 54 topics, evaluating at k=100...
<function columns at 0x7eca0a5bbba0>


webis/monoelectra-base rank: 0batch [00:00, ?batch/s]
/home/kpch/miniconda3/envs/air/lib/python3.12/site-packages/pyterrier/_evaluation/_validation.py:41: UserWarning: Transformer Cascade Model (BM25 >> Cross-Encoder) ((TerrierRetr(BM25) >> RankCutoff(1000) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x7eca0a4c0170> >> <lightning_ir_pyterrier.CrossEncoder object at 0x7ecb30d875f0> >> RankCutoff(100))) at position 1 failed to validate: Cannot determine outputs for (TerrierRetr(BM25) >> RankCutoff(1000) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x7eca0a4c0170> >> <lightning_ir_pyterrier.CrossEncoder object at 0x7ecb30d875f0> >> RankCutoff(100)) with inputs: ['qid', 'query'] - if your pipeline works, set validate='ignore' to remove this warning, or add transform_output method to the transformers in this pipeline to clarify how it works
  warn(


webis/monoelectra-base rank: 0batch [00:00, ?batch/s]


pt.Experiment:   0%|          | 0/2 [00:00<?, ?system/s]

19:25:34.083 [main] ERROR org.terrier.querying.LocalManager -- Problem running Matching, returning empty result set as query 1105792
java.io.IOException: Unknown field DEFINE - known fields are [text]
	at org.terrier.matching.matchops.SingleTermOp.getPostingIterator(SingleTermOp.java:120)
	at org.terrier.matching.matchops.SingleTermOp.getMatcher(SingleTermOp.java:149)
	at org.terrier.matching.PostingListManager.<init>(PostingListManager.java:304)
	at org.terrier.matching.PostingListManager.<init>(PostingListManager.java:282)
	at org.terrier.matching.daat.Full.match(Full.java:89)
	at org.terrier.querying.LocalManager$ApplyLocalMatching.process(LocalManager.java:518)
	at org.terrier.querying.LocalManager.runSearchRequest(LocalManager.java:895)


pt.Experiment:  50%|█████     | 1/2 [00:01<00:01,  1.78s/system]

19:25:35.608 [main] ERROR org.terrier.querying.LocalManager -- Problem running Matching, returning empty result set as query 1105792
java.io.IOException: Unknown field DEFINE - known fields are [text]
	at org.terrier.matching.matchops.SingleTermOp.getPostingIterator(SingleTermOp.java:120)
	at org.terrier.matching.matchops.SingleTermOp.getMatcher(SingleTermOp.java:149)
	at org.terrier.matching.PostingListManager.<init>(PostingListManager.java:304)
	at org.terrier.matching.PostingListManager.<init>(PostingListManager.java:282)
	at org.terrier.matching.daat.Full.match(Full.java:89)
	at org.terrier.querying.LocalManager$ApplyLocalMatching.process(LocalManager.java:518)
	at org.terrier.querying.LocalManager.runSearchRequest(LocalManager.java:895)
<function columns at 0x7eca0a5bbba0>


webis/monoelectra-base rank:   0%|          | 0/50575 [00:00<?, ?batch/s]


pt.Experiment: 100%|██████████| 2/2 [04:14<00:00, 127.24s/system]


In [14]:
experiment_results_at_k

,name,RR,nDCG@10,nDCG@100,AP@100
0,BM25 Baseline at 100,0.79310,0.480634,0.488215,0.301690
1,Cascade Model (BM25 >> Cross-Encoder),0.93696,0.743385,0.683329,0.507775


## Discussion

Use the following marked cells to respond to the following prompts

### 1. Desribe your cascade model and why you implemented it in the way you did.

(250 words max)



My cascade model is a two stage re-ranking pipeline with the following components:

**Stage 1: BM25**: I use BM25 which is a fast lexical sparse retriever. It scans the entire index to fetch the top 1000 documents. This stage is optimized for speed and recall, and retrieving 1000 documents so potentially relevant documents are not missed. Although 1000 is a relatively small number compared to the full corpus, it is a manageable size for the next stage.

**Stage 2: Cross-Encoder**: Those 1000 candidates are then passed directly to a Cross-Encoder (webis/monoelectra-base). This is a powerful and semantically aware model. It re-ranks all 1000 documents by jointly encoding the query and each document, allowing it to capture fine-grained relevance interactions. The final list is then truncated to the top 100 (% K_STAGE2_CANDIDATES) for evaluation.

I implemented this design because it's a standard and effective pattern that balances speed and accuracy. It uses a fast, "cheap" retriever (BM25) to do the heavy lifting of finding 1000 candidates from millions, and then applies the computationally expensive, high-quality re-ranker (the Cross-Encoder) to only that small and promising set.

### 2. Describe how you could improve your cascade model further.

(250 words max)

There are some ways to improve this cascade model further:

1. Introduce a Hybrid First Stage: The biggest improvement would be to implement the hybrid-stage pipeline. By adding a Bi-Encoder (like `webis/bert-bi-encoder`, `colbert` or `e5`) with the BM25 and then rerannking via Cross-Encoder. We could filter the 1000 candidates based on lexical (BM25, keywords) and semantic (bi-encoder) understanding before passing it through our cross-encoder reranker. The slow Cross-Encoder would then process on these more richer candidates, likely improving final ranking quality.

2. Tune the Candidate Set Size: We are re-ranking 1000 documents. We could experiment with this number (K_STAGE1_CANDIDATES). Perhaps 500 documents are enough, which would double the speed of the Cross-Encoder stage. Conversely, if accuracy is paramount, we might test 2000 candidates, assuming the Cross-Encoder can handle it.

3. Component Model Upgrades: We can replace the components with stronger models. Instead of `webis/monoelectra-base`, we could use a state-of-the-art model like `BAAI` or `t5` which could provide a significant quality boost. These models have shown strong performance on various benchmarks and could lead to better ranking quality.

4. Fine-tuning the Cross-Encoder: If we have domain-specific training data, fine-tuning the cross_encoder on that data would likely yield the largest improvement in ranking quality for that specific task. But this may reduce generalization.


### 3. Describe why you chose the evaluation measures to evalaute your cascade model.

(250 words max)

I chose the following evaluation measures to evaluate my cascade model in a comprehensive manner:

Below metrics evaluate the immediate and first page results:

- nDCG@10 (Normalized Discounted Cumulative Gain at 10): This checks if a result is relevant and how relevant it is (graded relevance). It strongly rewards placing highly relevant documents at the very top of the list.

- RR (Reciprocal Rank): This metric measures 'find-it-fast' quality. It focuses only on the rank of the single first relevant document. This is crucial for tasks where a user is looking for one specific correct answer.

Below metrics evaluate the entire set of 100 documents returned by the final cascade stage:
- nDCG@100: This extends the logic of nDCG to the full 100 results and measures the overall ranking quality of the entire list.

- MAP@100 (Mean Average Precision at 100): This metric evaluates the precision at each relevant document found within the top 100. It's great for measuring model's ability to find many relevant documents (recall) and rank all of them highly (precision) within the full list.

By using all four, we can measure the different aspects of the cascade model's effectiveness. The model is better at showing relevant results on the first page (nDCG@10), better at finding the first correct answer (RR), and better at ranking the entire list (nDCG@100, MAP@100).